In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 

import torchvision 
from torchvision.datasets import CIFAR10

from torch.utils.data import DataLoader
import torchvision.transforms as transforms


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))

])

In [4]:
train_set = CIFAR10(root="./",train=True,download=True,transform=transform)
test_set = CIFAR10(root="./",train=False,download=True,transform=transform)

In [5]:
train_set

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
train_loder = DataLoader(train_set,batch_size=64,shuffle=True)
test_loder = DataLoader(test_set,batch_size=64)

# BUILD THE CNN


In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size and stride 
            
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
            
        )

        self.fc_layer=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x= self.fc_layer(x)

        return x



In [8]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [9]:
print(next(model.parameters()).device)

cuda:0


# Training the CNN

In [10]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loder:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    epoch_training_loss /= len(train_loder)

    print(f"Epoch {epoch+1}, Loss = {epoch_training_loss:.4f}")



Epoch 1, Loss = 1.3691
Epoch 2, Loss = 0.9381
Epoch 3, Loss = 0.7552
Epoch 4, Loss = 0.6265
Epoch 5, Loss = 0.5195
Epoch 6, Loss = 0.4280
Epoch 7, Loss = 0.3447
Epoch 8, Loss = 0.2692
Epoch 9, Loss = 0.2071
Epoch 10, Loss = 0.1608


In [11]:

model.eval()

correct_labels = 0
total_labels = 0

with torch.no_grad():
    for images, labels in test_loder:
        images = images.to(device)
        labels = labels.to(device)

        output = model(images)
        _, predicted = torch.max(output, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = 100 * correct_labels / total_labels

print(f"Test Accuracy: {accuracy:.2f}%")



Test Accuracy: 76.36%
